In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import skew, kurtosis

In [38]:
"""def compute_statistics(window):
    stats = {}
    for col in window.columns:
        if window[col].dtype in ["int64", "float64"]:  # Numeric columns
            stats[f"{col}_MinV"] = np.min(window[col], axis = 0)
            stats[f"{col}_MaxV"] = np.max(window[col], axis = 0)
            stats[f"{col}_std"] = np.std(window[col], axis = 0)
            stats[f"{col}_AM"] = np.mean(window[col], axis = 0)
            stats[f"{col}_AAM"] = np.mean(np.abs(window[col]), axis = 0)
            #stats[f"{col}_GM"] = np.exp(np.mean(np.log(window[col] + 1e-9), axis = 0))
            stats[f"{col}_HM"] = len(window[col]) / np.sum(1.0 / (window[col] + 1e-9), axis=0)
            stats[f"{col}_Sum"] = np.sum(window[col], axis=0)
            stats[f"{col}_Q1"] = np.percentile(window[col], 25, axis=0)
            stats[f"{col}_Median"] = np.median(window[col], axis=0)
            stats[f"{col}_Q3"] = np.percentile(window[col], 75, axis=0)
            stats[f"{col}_Variance"] = np.var(window[col], axis=0)
            #stats[f"{col}_Skewness"] = skew(window[col], axis=0)
            #stats[f"{col}_Kurtosis"] = kurtosis(window[col], axis=0)
            stats[f"{col}_Zero_Crossings"] = np.sum(np.diff(np.sign(window[col]), axis=0) != 0, axis=0)
            stats[f"{col}_Mean_Energy"] = np.mean(window[col]**2, axis=0)
            stats[f"{col}_Mean_Curve_Length"] = np.mean(np.abs(np.diff(window[col], axis=0)), axis=0)
            #stats[f"{col}_Mean_Teager_Energy"] = np.mean(window[col][1:-1]**2 - window[col][:-2] * window[col][2:], axis=0)

    return pd.Series(stats)


def sliding_window(df, window_size, step_size):
    aggregated_windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        print(window.columns)
        #aggregated_window = compute_statistics(window)
        #aggregated_windows.append(aggregated_window)
    return pd.DataFrame(aggregated_windows)"""

In [111]:
def remove_outliers_iqr(df):
    df_cleaned = df.copy()

    for column in df_cleaned.columns:
        if pd.api.types.is_numeric_dtype(df_cleaned[column]):
            Q1 = df_cleaned[column].quantile(0.25)
            Q3 = df_cleaned[column].quantile(0.75)
            
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            df_cleaned = df_cleaned[(df_cleaned[column] >= lower_bound) & (df_cleaned[column] <= upper_bound)]
    
    return df_cleaned

In [4]:
labels_df = pd.read_csv("PHQ8_labels.csv")
participants = list(labels_df["Participant_ID"])
#participants = [304]
print(participants)

[300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 395, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492]


In [131]:
labels_df[labels_df["Participant_ID"] == 304]

,Participant_ID,PHQ8_Binary,PHQ8_Score
4,304,0,6


In [138]:

video_shape_list = []
audio_shape_list = []


for participant in participants:
    print(participant)
    # video
    au_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_AUs.txt")
    feat_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_features.txt")
    feat3d_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_features3D.txt")
    gaze_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_gaze.txt")
    pose_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_pose.txt")
    


    au_df.rename(columns = {" success" : " success0"}, inplace = True)
    feat_df.rename(columns = {" success" : " success1"}, inplace = True)
    feat3d_df.rename(columns = {" success" : " success2"}, inplace = True)
    gaze_df.rename(columns = {" success" : " success3"}, inplace = True)
    pose_df.rename(columns = {" success" : " success4"}, inplace = True)

    au_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    feat_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    feat3d_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    gaze_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)
    pose_df.drop(columns = [" timestamp", " confidence", "frame"], inplace = True)

    video_df = pd.concat([au_df, feat_df, feat3d_df, gaze_df, pose_df], axis = 1)
    
    video_df.drop(video_df.loc[video_df[" success0"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success1"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success2"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success3"]==0].index, inplace=True)
    video_df.drop(video_df.loc[video_df[" success4"]==0].index, inplace=True)
    
    video_df.drop(columns = [" success0", " success1", " success2", " success3", " success4"], inplace = True)
    

    video_df = remove_outliers_iqr(video_df)
    video_df.reset_index(drop = True, inplace = True)
    print("video:", video_df.shape)


    # audio
    formant_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_FORMANT.csv", header = None)
    formant_df.columns = ["F0", "F1", "F2", "F3", "F4"]
    covarep_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_COVAREP.csv", header = None)

    audio_df = pd.concat([formant_df, covarep_df], axis = 1)
    audio_df.dropna(inplace = True)

    audio_df.drop(audio_df.loc[audio_df[1]==0].index, inplace=True)
    audio_df.drop(columns=[1], inplace = True, axis = 1)

    audio_df = remove_outliers_iqr(audio_df)
    audio_df.reset_index(drop = True, inplace = True)
    print("audio:", audio_df.shape)


    # matching the sequential rows
    merged_df = pd.concat([video_df, audio_df], axis = 1)
    merged_df.dropna(inplace = True)

    video_df = merged_df.iloc[:, :378]
    # adding target label
    video_df["Target"] = labels_df[labels_df["Participant_ID"] == participant].iloc[0,1]

    audio_df = merged_df.iloc[:, 378:]
    audio_df["Target"] = labels_df[labels_df["Participant_ID"] == participant].iloc[0,1]


    # saving csvs

    video_df.to_csv(fr"video_matchingrows\{participant}_CLNF.csv", index = False)
    audio_df.to_csv(fr"audio_matchingrows\{participant}.csv", index = False)



    print("video after split:", video_df.shape)
    print("audio after split:", audio_df.shape)



300
video: (2023, 378)
audio: (7059, 78)
video after split: (2023, 379)
audio after split: (2023, 79)
301
video: (4557, 378)
audio: (3950, 78)
video after split: (3950, 379)
audio after split: (3950, 79)
302
video: (4026, 378)
audio: (1371, 78)
video after split: (1371, 379)
audio after split: (1371, 79)
303
video: (6729, 378)
audio: (6368, 78)
video after split: (6368, 379)
audio after split: (6368, 79)
304
video: (1613, 378)
audio: (2291, 78)
video after split: (1613, 379)
audio after split: (1613, 79)
305
video: (4115, 378)
audio: (6777, 78)
video after split: (4115, 379)
audio after split: (4115, 79)
306
video: (4888, 378)
audio: (7217, 78)
video after split: (4888, 379)
audio after split: (4888, 79)
307
video: (6875, 378)
audio: (7190, 78)
video after split: (6875, 379)
audio after split: (6875, 79)
308
video: (4841, 378)
audio: (2428, 78)
video after split: (2428, 379)
audio after split: (2428, 79)
309
video: (4069, 378)
audio: (1081, 78)
video after split: (1081, 379)
audio afte

C:\Users\sachl\AppData\Local\Temp\ipykernel_8644\1287349743.py:9: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  feat_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_features.txt")
C:\Users\sachl\AppData\Local\Temp\ipykernel_8644\1287349743.py:10: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75

video: (6536, 378)
audio: (7567, 78)
video after split: (6536, 379)
audio after split: (6536, 79)
368
video: (12209, 378)
audio: (7476, 78)
video after split: (7476, 379)
audio after split: (7476, 79)
369
video: (9859, 378)
audio: (7511, 78)
video after split: (7511, 379)
audio after split: (7511, 79)
370
video: (6680, 378)
audio: (6978, 78)
video after split: (6680, 379)
audio after split: (6680, 79)
371
video: (4759, 378)
audio: (3086, 78)
video after split: (3086, 379)
audio after split: (3086, 79)
372
video: (4955, 378)
audio: (4245, 78)
video after split: (4245, 379)
audio after split: (4245, 79)
373
video: (5537, 378)
audio: (13155, 78)
video after split: (5537, 379)
audio after split: (5537, 79)
374
video: (7991, 378)
audio: (4876, 78)
video after split: (4876, 379)
audio after split: (4876, 79)
375
video: (3261, 378)
audio: (1618, 78)
video after split: (1618, 379)
audio after split: (1618, 79)
376
video: (1029, 378)
audio: (3713, 78)
video after split: (1029, 379)
audio after 

C:\Users\sachl\AppData\Local\Temp\ipykernel_8644\1287349743.py:9: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  feat_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_features.txt")
C:\Users\sachl\AppData\Local\Temp\ipykernel_8644\1287349743.py:10: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75

video: (3467, 378)
audio: (2407, 78)
video after split: (2407, 379)
audio after split: (2407, 79)
397
video: (10071, 378)
audio: (3158, 78)
video after split: (3158, 379)
audio after split: (3158, 79)
399
video: (2307, 378)
audio: (4398, 78)
video after split: (2307, 379)
audio after split: (2307, 79)
400
video: (3325, 378)
audio: (1802, 78)
video after split: (1802, 379)
audio after split: (1802, 79)
401
video: (7730, 378)
audio: (3807, 78)
video after split: (3807, 379)
audio after split: (3807, 79)
402
video: (7499, 378)
audio: (2098, 78)
video after split: (2098, 379)
audio after split: (2098, 79)
403
video: (3130, 378)
audio: (6000, 78)
video after split: (3130, 379)
audio after split: (3130, 79)
404
video: (4168, 378)
audio: (2257, 78)
video after split: (2257, 379)
audio after split: (2257, 79)
405
video: (6440, 378)
audio: (8517, 78)
video after split: (6440, 379)
audio after split: (6440, 79)
406
video: (3300, 378)
audio: (3384, 78)
video after split: (3300, 379)
audio after s

C:\Users\sachl\AppData\Local\Temp\ipykernel_8644\1287349743.py:9: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  feat_df = pd.read_csv(fr"datasets\DAIC-Woz\{participant}_P\{participant}_CLNF_features.txt")
C:\Users\sachl\AppData\Local\Temp\ipykernel_8644\1287349743.py:10: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75

video: (4090, 378)
audio: (2301, 78)
video after split: (2301, 379)
audio after split: (2301, 79)
433
video: (1662, 378)
audio: (3040, 78)
video after split: (1662, 379)
audio after split: (1662, 79)
434
video: (10597, 378)
audio: (8136, 78)
video after split: (8136, 379)
audio after split: (8136, 79)
435
video: (1586, 378)
audio: (3647, 78)
video after split: (1586, 379)
audio after split: (1586, 79)
436
video: (2826, 378)
audio: (4089, 78)
video after split: (2826, 379)
audio after split: (2826, 79)
437
video: (2546, 378)
audio: (2565, 78)
video after split: (2546, 379)
audio after split: (2546, 79)
438
video: (7258, 378)
audio: (4318, 78)
video after split: (4318, 379)
audio after split: (4318, 79)
439
video: (10120, 378)
audio: (5972, 78)
video after split: (5972, 379)
audio after split: (5972, 79)
440
video: (9437, 378)
audio: (8439, 78)
video after split: (8439, 379)
audio after split: (8439, 79)
441
video: (7232, 378)
audio: (1637, 78)
video after split: (1637, 379)
audio after 

In [2]:
import os

In [3]:
audio_path = r"audio_matchingrows"
video_path = r"video_matchingrows"

In [ ]:
# saving audio combined csv

In [5]:
complete_audio_clean_matchingrows = pd.DataFrame()

for participant in participants:
    path = os.path.join(audio_path, fr"{participant}.csv")
    temp = pd.read_csv(path)


    complete_audio_clean_matchingrows = pd.concat([complete_audio_clean_matchingrows, temp], ignore_index=True, axis = 0)
    print(complete_audio_clean_matchingrows.shape)

(2023, 79)
(5973, 79)
(7344, 79)
(13712, 79)
(15325, 79)
(19440, 79)
(24328, 79)
(31203, 79)
(33631, 79)
(34712, 79)
(36151, 79)
(37566, 79)
(38811, 79)
(40364, 79)
(46832, 79)
(50488, 79)
(52305, 79)
(53344, 79)
(54696, 79)
(55921, 79)
(57345, 79)
(59543, 79)
(62419, 79)
(63830, 79)
(66414, 79)
(68532, 79)
(69755, 79)
(71543, 79)
(74814, 79)
(76367, 79)
(77563, 79)
(78827, 79)
(81074, 79)
(83454, 79)
(86653, 79)
(89454, 79)
(90907, 79)
(97273, 79)
(98615, 79)
(101913, 79)
(103415, 79)
(105957, 79)
(107586, 79)
(111538, 79)
(115806, 79)
(118418, 79)
(119870, 79)
(122022, 79)
(124956, 79)
(127310, 79)
(131942, 79)
(136017, 79)
(137284, 79)
(138800, 79)
(142992, 79)
(146820, 79)
(147522, 79)
(148739, 79)
(150782, 79)
(152414, 79)
(153841, 79)
(155205, 79)
(160939, 79)
(169290, 79)
(170165, 79)
(174215, 79)
(180751, 79)
(188227, 79)
(195738, 79)
(202418, 79)
(205504, 79)
(209749, 79)
(215286, 79)
(220162, 79)
(221780, 79)
(222809, 79)
(227925, 79)
(230886, 79)
(234026, 79)
(243027, 79)
(2

In [8]:
display(complete_audio_clean_matchingrows)

,F0,F1,F2,F3,F4,0,2,3,4,5,...,65,66,67,68,69,70,71,72,73,Target
0,293.20,1635.5,2538.1,3242.0,3988.0,269.0,0.038446,0.111760,0.9758,0.22440,...,-0.92749,-0.74712,-0.54045,-0.42579,-0.40818,-0.27205,-0.18758,-0.115740,-0.079553,0
1,318.92,1656.0,2726.0,3413.0,4225.6,269.0,0.025810,0.080351,0.0000,0.34022,...,-0.93171,-0.74674,-0.55854,-0.43645,-0.41822,-0.27231,-0.18013,-0.141710,-0.093744,0
2,582.25,1469.5,2423.0,3034.3,4043.2,271.0,0.045581,0.179910,-5.3092,0.47652,...,-0.98807,-0.77188,-0.59350,-0.41623,-0.29127,-0.19868,-0.11678,-0.124110,-0.113760,0
3,476.97,2103.9,3067.2,4082.9,4639.4,231.0,0.058752,0.170800,-4.2914,0.26402,...,-0.66118,-0.66544,-0.68464,-0.62956,-0.61106,-0.35705,-0.26494,-0.145970,-0.166570,0
4,507.50,1305.0,1740.2,2812.2,3542.5,182.0,0.082644,0.237140,-4.6301,0.67685,...,-0.70488,-0.71061,-0.69635,-0.60842,-0.58210,-0.39583,-0.28554,-0.204080,-0.150490,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568340,222.24,1945.2,2819.0,4040.8,4784.6,152.0,0.201670,0.548830,9.2067,0.35271,...,-0.88044,-0.56210,-0.64258,-0.57587,-0.43868,-0.30497,-0.28417,-0.144310,-0.051550,0
568341,581.16,1820.0,2496.1,3225.3,4017.1,209.0,0.034666,0.349750,5.6566,0.12914,...,-0.84567,-0.73813,-0.63178,-0.54548,-0.51871,-0.30890,-0.27262,-0.134150,-0.067753,0
568342,523.00,1251.3,2007.7,2864.9,3914.0,207.5,0.134000,0.386820,-5.1442,0.19695,...,-0.83250,-0.73273,-0.62752,-0.53681,-0.51673,-0.27603,-0.26620,-0.111950,-0.042231,0
568343,523.00,852.0,1765.2,2886.5,4060.9,206.0,0.162830,0.502370,10.1350,0.21850,...,-0.86057,-0.74837,-0.61421,-0.49968,-0.47533,-0.19490,-0.19090,-0.020043,0.059151,0


In [9]:
complete_audio_clean_matchingrows["Target"].value_counts()

Target
0    397624
1    170721
Name: count, dtype: int64

In [6]:
complete_audio_clean_matchingrows.to_csv("complete_audio_clean_matchingrows.csv", index = False)

In [7]:
# saving video combined csv

In [10]:
complete_video_clean_matchingrows = pd.DataFrame()


for participant in participants:
    print(participant)
    path = os.path.join(video_path, fr"{participant}_CLNF.csv")
    temp = pd.read_csv(path)


    complete_video_clean_matchingrows = pd.concat([complete_video_clean_matchingrows, temp], ignore_index=True, axis = 0)
    print(complete_video_clean_matchingrows.shape)

300
(2023, 379)
301
(5973, 379)
302
(7344, 379)
303
(13712, 379)
304
(15325, 379)
305
(19440, 379)
306
(24328, 379)
307
(31203, 379)
308
(33631, 379)
309
(34712, 379)
310
(36151, 379)
311
(37566, 379)
312
(38811, 379)
313
(40364, 379)
314
(46832, 379)
315
(50488, 379)
316
(52305, 379)
317
(53344, 379)
318
(54696, 379)
319
(55921, 379)
320
(57345, 379)
321
(59543, 379)
322
(62419, 379)
323
(63830, 379)
324
(66414, 379)
325
(68532, 379)
326
(69755, 379)
327
(71543, 379)
328
(74814, 379)
329
(76367, 379)
330
(77563, 379)
331
(78827, 379)
332
(81074, 379)
333
(83454, 379)
334
(86653, 379)
335
(89454, 379)
336
(90907, 379)
337
(97273, 379)
338
(98615, 379)
339
(101913, 379)
340
(103415, 379)
341
(105957, 379)
343
(107586, 379)
344
(111538, 379)
345
(115806, 379)
346
(118418, 379)
347
(119870, 379)
348
(122022, 379)
349
(124956, 379)
350
(127310, 379)
351
(131942, 379)
352
(136017, 379)
353
(137284, 379)
354
(138800, 379)
355
(142992, 379)
356
(146820, 379)
357
(147522, 379)
358
(148739, 379

In [11]:
display(complete_video_clean_matchingrows)

,AU01_r,AU02_r,AU04_r,AU05_r,AU06_r,AU09_r,AU10_r,AU12_r,AU14_r,AU15_r,...,x_h1,y_h1,z_h1,Tx,Ty,Tz,Rx,Ry,Rz,Target
0,0.000000,0.0,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.169596,0.411488,-0.895497,24.34710,47.5664,535.164,0.198389,0.055638,-0.117204,0
1,0.000000,0.0,0.574712,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.156311,0.412949,-0.897240,24.00500,47.6058,534.351,0.202752,0.053431,-0.116840,0
2,0.000000,0.0,0.516607,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.144607,0.401512,-0.904365,11.48700,47.4767,533.786,0.205838,0.078339,-0.136158,0
3,0.000000,0.0,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.143316,0.440597,-0.886191,8.98588,50.1869,533.380,0.236301,0.087538,-0.146588,0
4,0.000000,0.0,0.508391,0.0,0.0,0.53615,0.0,0.0,0.0,0.0,...,-0.166556,0.422581,-0.890890,9.34586,50.1033,534.246,0.226615,0.091649,-0.148060,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568340,0.520283,0.0,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.445046,0.190221,-0.875072,100.78900,80.4818,612.787,0.102507,0.052492,0.078945,0
568341,0.000000,0.0,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.446359,0.180351,-0.876491,100.99200,80.6265,613.252,0.100756,0.054022,0.079229,0
568342,0.000000,0.0,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.408466,0.178801,-0.895090,101.43900,80.6040,614.223,0.099579,0.048727,0.076895,0
568343,0.000000,0.0,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,-0.405196,0.182628,-0.895803,101.38300,80.5637,613.862,0.101123,0.047389,0.076082,0


In [12]:
complete_video_clean_matchingrows["Target"].value_counts()

Target
0    397624
1    170721
Name: count, dtype: int64

In [13]:
complete_video_clean_matchingrows.to_csv("complete_video_clean_matchingrows.csv", index = False)